In [49]:
# notebooks/01_tax_optimization_demo.ipynb
# Run cells in order. Install deps first: pip install taxopt[notebook]

In [50]:
# Cell 1 — Imports
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf
from datetime import date
from dataclasses import dataclass, replace
from typing import cast, Dict
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import time

from taxopt import (
    Portfolio, TaxLot,
    USCapitalGainsPolicy, LotMethod,
    CvxpyOptimizer, PortfolioPolicy, OptimizationInputs,
    LotClose, LongOpen, ShortOpen,
    TaxReport,
)


In [51]:
# Cell 2 — Download price data
# TICKERS: list[str] = [
#     "NVDA", "AAPL", "MSFT", "AMZN", "GOOGL", "AVGO", "GOOG", "META", "TSLA", "BRK-B",
#     "JPM", "LLY", "XOM", "JNJ", "WMT", "V", "MU", "COST", "MA", "NFLX",
#     "ABBV", "CVX", "PLTR", "PG", "HD", "CAT", "AMD", "GE", "BAC", "CSCO",
# ]
TICKERS: list[str] = [
    "VTI", "SPY", "QQQ", "SCHD", "IJR", "VUG", "VTV", "IVE", "IVW", "VO",
    "VEA", "VWO", "IEFA", "EEM", "VXUS", "EWJ", "EZU", "MCHI", "AGG", "BND",
    "BNDX", "SHY", "TLT", "LQD", "VNQ", "GLD", "DBC", "MLPA", "VGT", "XLV",
]
n: int = len(TICKERS)

_raw = yf.download(TICKERS, start="2015-01-01", end="2026-02-28", auto_adjust=True)
assert _raw is not None

raw: pd.DataFrame = cast(pd.DataFrame, _raw["Close"])[TICKERS].dropna()
returns: pd.DataFrame = cast(pd.DataFrame, raw.pct_change().dropna())

print(f"Price data: {raw.index[0].date()} → {raw.index[-1].date()}, {len(raw)} days")
raw.tail(3)


[*********************100%***********************]  30 of 30 completed


Price data: 2015-01-02 → 2026-02-27, 2805 days


Ticker,VTI,SPY,QQQ,SCHD,IJR,VUG,VTV,IVE,IVW,VO,...,BNDX,SHY,TLT,LQD,VNQ,GLD,DBC,MLPA,VGT,XLV
Date,,,,,,,,,,,,,,,,,,,,,
2026-02-25,341.829987,691.262207,616.679993,31.51,130.453842,470.320007,206.649994,220.765610,122.135574,304.529999,...,48.930344,82.817375,89.612022,111.267105,94.870003,473.420013,24.75,52.950001,751.260010,157.830002
2026-02-26,340.489990,687.422607,609.239990,31.51,131.182678,465.459991,206.929993,221.124283,120.586899,307.059998,...,49.010185,82.847290,89.970818,111.336868,95.519997,477.480011,24.76,53.299999,740.190002,157.419998
2026-02-27,338.769989,684.121643,607.289978,31.77,129.495346,460.869995,207.259995,221.273727,119.457863,306.200012,...,49.100002,82.957001,90.518997,111.297005,95.690002,483.750000,25.10,53.509998,726.700012,160.199997


In [52]:
# Cell 3 — Helper functions
def get_prices(as_of: date) -> dict[str, float]:
    ts = raw.index[raw.index <= pd.Timestamp(as_of)]
    row = raw.iloc[0] if len(ts) == 0 else raw.loc[ts[-1]]
    return {t: float(row[t]) for t in TICKERS}


def compute_market_betas(
    lookback_end: date,
    lookback_days: int = 252,
) -> Dict[str, float]:
    """
    Estimate per-stock market betas vs an equal-weight market index
    built from the current TICKERS universe, over a rolling window.
    """
    end_ts = pd.Timestamp(lookback_end)
    win = returns.loc[end_ts - pd.Timedelta(days=lookback_days * 2) : end_ts].dropna(how="any")
    if len(win) < lookback_days:
        raise ValueError(f"Insufficient history on {lookback_end}: {len(win)} days")

    # Use last 'lookback_days' rows
    win = win.tail(lookback_days)

    # Equal-weight market index returns
    r_m = win.mean(axis=1).to_numpy(dtype=float)  # shape (T,)

    # Centered to avoid intercept leakage
    r_m_centered = r_m - r_m.mean()
    denom = float((r_m_centered ** 2).sum())
    if denom < 1e-10:
        # Degenerate case; return zeros
        return {t: 0.0 for t in TICKERS}

    betas: Dict[str, float] = {}
    for t in TICKERS:
        r_i = win[t].to_numpy(dtype=float)
        r_i_centered = r_i - r_i.mean()
        num = float((r_i_centered * r_m_centered).sum())
        betas[t] = num / denom

    return betas


def get_inputs(
    as_of: date,
    policy: PortfolioPolicy,
    lookback_days: int = 252,
) -> OptimizationInputs:
    end  = pd.Timestamp(as_of)
    rets = returns.loc[end - pd.Timedelta(days=lookback_days * 2) : end].dropna(how="any")
    if len(rets) < lookback_days:
        raise ValueError(f"Insufficient history on {as_of}: {len(rets)} days")

    cov: np.ndarray = rets.tail(lookback_days).cov().to_numpy() * 252 + np.eye(n) * 1e-6

    # Alpha: 12-1 month momentum → rank → z-score → beta-neutralize → risk-normalize
    momentum = (1 + rets.iloc[-252:-20]).prod() - 1
    ranked   = momentum.rank().to_numpy(dtype=float)
    w        = (ranked - ranked.mean()) / (ranked.std() + 1e-8)  # standardized signal

    betas_dict = compute_market_betas(as_of, lookback_days=lookback_days)
    b = np.array([betas_dict[t] for t in TICKERS], dtype=float)

    # scalar OLS coefficient a for regression w = a * b + residual
    num = float(w @ b)
    den = float(b @ b) if float(b @ b) > 1e-8 else 1.0
    a   = num / den
    v   = w - a * b    # residual weights, now market-beta-neutral

    v /= np.sqrt(max(float(v @ cov @ v), 1e-8))
    alpha = {t: float((cov @ v)[i]) for i, t in enumerate(TICKERS)}

    return OptimizationInputs(
        alpha=alpha,
        covariance=cov,
        assets=TICKERS,
        prices=get_prices(as_of),
        as_of=as_of,
        risk_aversion=policy.risk_aversion,
        tax_aversion=policy.tax_aversion,
        gross_leverage=policy.gross_leverage,
        net_exposure=policy.net_exposure,
        max_weight=policy.max_weight,
        max_turnover=policy.max_turnover,
    )


@dataclass
class TaxLedger:
    """
    Tracks cumulative realized gains/losses over the backtest.
    Assumes the investor has external gains to absorb harvested losses
    immediately — standard assumption in the TLH literature.
    No year-end reset, no carryforward complexity.
    """
    st_realized: float = 0.0
    lt_realized: float = 0.0

    def record(self, report: TaxReport) -> None:
        self.st_realized += report.totals_by_type.get("short_term", 0.0)
        self.lt_realized += report.totals_by_type.get("long_term",  0.0)

    def cumulative_tax_value(self, policy: USCapitalGainsPolicy) -> float:
        """Net tax impact of all realized activity. Negative = net tax saving."""
        return self.st_realized * policy.st_rate + self.lt_realized * policy.lt_rate

    def after_tax_nav(
        self,
        portfolio: Portfolio,
        prices: dict[str, float],
        as_of: date,
        policy: USCapitalGainsPolicy,
    ) -> float:
        """
        Pre-tax NAV
        − DTL on unrealized gains (hypothetical liquidation tax today)
        − cumulative tax on realized gains (positive = owed, negative = saved)
        """
        unreal_st, unreal_lt = 0.0, 0.0
        for asset, lots in portfolio.lots.items():
            px = prices[asset]
            for lot in lots:
                if lot.quantity <= 0:
                    continue
                gain = (px - lot.cost_basis) * lot.quantity
                days = (as_of - lot.acquisition_date).days
                if days >= policy.lt_threshold_days:
                    unreal_lt += gain
                else:
                    unreal_st += gain

        dtl = unreal_st * policy.st_rate + unreal_lt * policy.lt_rate
        return portfolio.total_value(prices) - dtl - self.cumulative_tax_value(policy)

In [53]:
# Cell 4 — rebalance loop with ledger
POLICY = USCapitalGainsPolicy(lot_method=LotMethod.MIN_GAIN)

# ── Portfolio policy ──────────────────────────────────────────
POLICY_OPT = PortfolioPolicy(
    risk_aversion  = 2.0,
    tax_aversion   = 1.0,
    gross_leverage = 3.0,   # L + S  (e.g. L130/S30 → 1.6)
    net_exposure   = 1.0,   # L - S
    max_weight     = 0.25,
    max_turnover   = 0.50,  # None = unlimited
)

# ── Solver settings ───────────────────────────────────────────
SOLVER = CvxpyOptimizer(
    solver="SCIP",
    verbose=False,
    tax_aware=True,                 # enable tax-aware optimization (enable objective term)
    relax_turnover=True,            # gradually widens turnover if infeasible
    turnover_relax_step=0.05,       # +5pp per attempt
    turnover_relax_max_attempts=5,  # up to +25pp before giving up
    mip_gap=0.0,                   # 5% MIP gap for faster solve at the cost of optimality guarantee
)

# All month-ends in range; rebalance on each except the last,
# which serves only as the EOM date of the final holding period.
all_dates: list[date] = pd.date_range("2017-12-30", "2026-02-28", freq="ME").date.tolist()
rebal_dates = all_dates[:-1]   # rebalance dates: Dec 2022 … Jan 2026
eom_dates   = all_dates[1:]    # EOM dates:       Jan 2023 … Feb 2026

portfolio = Portfolio(cash=100_000.0)
ledger    = TaxLedger()

# Synthetic first row: initial cash at the first rebal date (before any trades)
history: list[dict] = [{
    "date":          rebal_dates[0],
    "nav_eom":       portfolio.cash,
    "after_tax_nav": portfolio.cash,
    "st_realized":   0.0,
    "lt_realized":   0.0,
    "st_cumulative": 0.0,
    "lt_cumulative": 0.0,
    "tax_alpha":     0.0,
    "num_actions":   0,
    "num_lots":      0,
}]

for i, (rebal_date, eom_date) in enumerate(zip(rebal_dates, eom_dates)):
    is_first   = (i == 0)

    # --- Prices and NAV at the start of the period (BOM / previous EOM) ---
    prices_bom = get_prices(rebal_date)
    tv_pre     = portfolio.total_value(prices_bom)

    # --- Optimization inputs ---
    inputs = get_inputs(
        rebal_date,
        policy=replace(POLICY_OPT, max_turnover=None if is_first else POLICY_OPT.max_turnover),
    )

    # --- Optimize and apply trades at current month-end prices ---
    t0     = time.perf_counter()
    result = SOLVER.solve(portfolio, inputs, POLICY, tv_pre)
    elapsed = time.perf_counter() - t0

    if result.status not in ("optimal", "optimal_inaccurate"):
        print(f"{rebal_date}: {result.status}, skipping")
        continue

    portfolio, tax_report = portfolio.apply_actions(result.actions, prices_bom, POLICY, rebal_date)
    tv_post = portfolio.total_value(prices_bom)
    assert abs(tv_post - tv_pre) / tv_pre < 1e-4, f"NAV not conserved: {tv_pre:.2f} → {tv_post:.2f}"

    # --- Tax alpha this period: tax saving as fraction of NAV ---
    period_tax_impact = (
        tax_report.totals_by_type.get("short_term", 0.0) * POLICY.st_rate +
        tax_report.totals_by_type.get("long_term",  0.0) * POLICY.lt_rate
    )
    tax_alpha = -period_tax_impact / tv_pre

    # --- EOM of this holding period ---
    prices_eom = get_prices(eom_date)
    tv_eom     = portfolio.total_value(prices_eom)

    # --- Record realized gains into ledger AFTER computing period tax_alpha ---
    ledger.record(tax_report)
    at_nav_eom = ledger.after_tax_nav(portfolio, prices_eom, eom_date, POLICY)

    # --- Append row indexed by the EOM date ---
    history.append({
        "date":          eom_date,
        "nav_eom":       tv_eom,
        "after_tax_nav": at_nav_eom,
        "st_realized":   tax_report.totals_by_type.get("short_term", 0.0),
        "lt_realized":   tax_report.totals_by_type.get("long_term",  0.0),
        "st_cumulative": ledger.st_realized,
        "lt_cumulative": ledger.lt_realized,
        "tax_alpha":     tax_alpha,
        "num_actions":   len(result.actions),
        "num_lots":      sum(len(v) for v in portfolio.lots.values()),
        "solve_time":    elapsed,
        "eff_turnover":  result.effective_turnover,
    })

    to_str = f"{result.effective_turnover:.0%}"
    turnover_note = (
        f" [relaxed → cap={result.effective_turnover:.0%}]"
        if (inputs.max_turnover is not None
            and result.effective_turnover > inputs.max_turnover + 1e-4)
        else ""
    )
    print(
        f"{eom_date} EOM=${tv_eom:>10,.0f} AT-NAV=${at_nav_eom:>10,.0f} "
        f"ST={tax_report.totals_by_type.get('short_term', 0.0):>+8,.0f} "
        f"LT={tax_report.totals_by_type.get('long_term',  0.0):>+8,.0f} "
        f"TaxAlpha={tax_alpha*100:>+6.3f}% "
        f"TO={to_str} "
        f"[{elapsed:.1f}s]{turnover_note}"
    )

df = pd.DataFrame(history).set_index("date")


2018-01-31 EOM=$   110,628 AT-NAV=$   106,446 ST=      +0 LT=      +0 TaxAlpha=-0.000% TO=150% [1.2s]
2018-02-28 EOM=$   108,785 AT-NAV=$   107,865 ST=  -2,420 LT=      +0 TaxAlpha=+0.766% TO=50% [0.5s]
2018-03-31 EOM=$   106,078 AT-NAV=$   106,175 ST=  -1,523 LT=      +0 TaxAlpha=+0.490% TO=50% [0.7s]
2018-04-30 EOM=$   103,398 AT-NAV=$   103,871 ST=     +35 LT=      +0 TaxAlpha=-0.011% TO=34% [0.5s]
2018-05-31 EOM=$   107,217 AT-NAV=$   105,756 ST=    -813 LT=      +0 TaxAlpha=+0.275% TO=50% [0.4s]
2018-06-30 EOM=$   104,529 AT-NAV=$   103,801 ST=    -670 LT=      +0 TaxAlpha=+0.219% TO=35% [0.8s]
2018-07-31 EOM=$   104,705 AT-NAV=$   103,094 ST=    -795 LT=      +0 TaxAlpha=+0.266% TO=50% [6.8s]
2018-08-31 EOM=$   112,225 AT-NAV=$   108,454 ST=  -1,661 LT=      +0 TaxAlpha=+0.555% TO=50% [0.8s]
2018-09-30 EOM=$   112,749 AT-NAV=$   109,772 ST=  -1,459 LT=      +0 TaxAlpha=+0.455% TO=50% [0.7s]
2018-10-31 EOM=$   107,795 AT-NAV=$   109,581 ST=     -16 LT=      +0 TaxAlpha=+0.005% TO=

In [54]:
# Cell 5 — Summary metrics
final_nav       = float(df["after_tax_nav"].iloc[-1])
pretax_nav      = float(df["nav_eom"].iloc[-1])
initial_nav     = float(df["nav_eom"].iloc[0])
total_st        = float(df["st_realized"].sum())
total_lt        = float(df["lt_realized"].sum())
total_realized  = total_st + total_lt
cum_tax         = ledger.cumulative_tax_value(POLICY)
pretax_return   = (pretax_nav  / initial_nav - 1) * 100
aftertax_return = (final_nav   / initial_nav - 1) * 100
tax_drag        = pretax_return - aftertax_return
tax_efficiency  = aftertax_return / pretax_return if pretax_return != 0 else float("nan")
tax_alpha_ann   = float(np.prod(df["tax_alpha"].to_numpy(dtype=float) + 1)) ** (12 / len(df)) - 1
n_years         = len(rebal_dates) / 12
pretax_cagr     = (pretax_nav  / initial_nav) ** (1 / n_years) - 1   # ← annualized
aftertax_cagr   = (final_nav   / initial_nav) ** (1 / n_years) - 1   # ← annualized

print("=" * 52)
print(f"  Backtest: {rebal_dates[0]} → {rebal_dates[-1]}  ({n_years:.1f} yrs)")
print("=" * 52)
print(f"  {'Initial NAV:':<32} ${initial_nav:>10,.0f}")
print(f"  {'Pre-tax final NAV:':<32} ${pretax_nav:>10,.0f}")
print(f"  {'After-tax final NAV:':<32} ${final_nav:>10,.0f}")
print()
print(f"  {'Pre-tax return:':<32} {pretax_return:>10.2f}%")
print(f"  {'Pre-tax CAGR:':<32} {pretax_cagr*100:>10.2f}%")   # ← added
print(f"  {'After-tax return:':<32} {aftertax_return:>10.2f}%")
print(f"  {'After-tax CAGR:':<32} {aftertax_cagr*100:>10.2f}%")  # ← added
print(f"  {'Tax drag:':<32} {tax_drag:>10.2f}%")
print(f"  {'Tax efficiency ratio:':<32} {tax_efficiency:>10.2%}")
print(f"  {'Annualized tax alpha:':<32} {tax_alpha_ann*100:>10.3f}%")
print()
print(f"  {'Cumul. ST realized:':<32} ${total_st:>10,.0f}")
print(f"  {'Cumul. LT realized:':<32} ${total_lt:>10,.0f}")
print(f"  {'Cumul. net realized:':<32} ${total_realized:>10,.0f}")
print(f"  {'Net tax impact (realized):':<32} ${cum_tax:>10,.0f}")
print(f"  {'  (neg = net tax saving)'}")
print()
print(f"  {'Effective rate on realized:':<32} {cum_tax/max(abs(total_realized),1)*100:>10.2f}%")
print(f"  {'Avg actions per rebalance:':<32} {df['num_actions'].mean():>10.1f}")
print(f"  {'Avg solve time:':<32} {df['solve_time'].mean():>10.2f}s")  # ← added since it's in df
print(f"  {'Final lot count:':<32} {int(df['num_lots'].iloc[-1]):>10d}")
print("=" * 52)

  Backtest: 2017-12-31 → 2026-01-31  (8.2 yrs)
  Initial NAV:                     $   100,000
  Pre-tax final NAV:               $   268,104
  After-tax final NAV:             $   257,008

  Pre-tax return:                      168.10%
  Pre-tax CAGR:                         12.84%
  After-tax return:                    157.01%
  After-tax CAGR:                       12.25%
  Tax drag:                             11.10%
  Tax efficiency ratio:                93.40%
  Annualized tax alpha:                 3.683%

  Cumul. ST realized:              $  -134,734
  Cumul. LT realized:              $     5,597
  Cumul. net realized:             $  -129,137
  Net tax impact (realized):       $   -46,037
    (neg = net tax saving)

  Effective rate on realized:          -35.65%
  Avg actions per rebalance:             52.4
  Avg solve time:                        1.60s
  Final lot count:                         54


In [55]:
# Cell 6 — Interactive charts
dates = df.index.astype(str).tolist()

# ── Chart 1: NAV vs After-Tax NAV ──────────────────────────────────────────
fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=dates, y=df["nav_eom"], name="Pre-Tax NAV",
                          line=dict(color="#3498db"), mode="lines+markers",
                          hovertemplate="%{x}<br>Pre-Tax NAV: $%{y:,.0f}<extra></extra>"))
fig1.add_trace(go.Scatter(x=dates, y=df["after_tax_nav"], name="After-Tax NAV",
                          line=dict(color="#2ecc71"), mode="lines+markers",
                          hovertemplate="%{x}<br>After-Tax NAV: $%{y:,.0f}<extra></extra>"))
fig1.update_layout(title="NAV vs After-Tax NAV (EOM)", hovermode="x unified",
                   xaxis_title="Date", yaxis_title="$ Value")
fig1.show()

# ── Chart 2: Realized ST / LT Gains per Rebalance ─────────────────────────
fig2 = go.Figure()
fig2.add_trace(go.Bar(x=dates, y=df["st_realized"], name="ST Realized",
                      marker_color="#e74c3c",
                      hovertemplate="%{x}<br>ST: $%{y:,.0f}<extra></extra>"))
fig2.add_trace(go.Bar(x=dates, y=df["lt_realized"], name="LT Realized",
                      marker_color="#2ecc71",
                      hovertemplate="%{x}<br>LT: $%{y:,.0f}<extra></extra>"))
fig2.update_layout(title="Realized ST / LT Gains per Rebalance",
                   barmode="group", hovermode="x unified",
                   xaxis_title="Date", yaxis_title="$ Gain / Loss")
fig2.show()

# ── Chart 3: Cumulative Realized ST / LT ──────────────────────────────────
fig3 = go.Figure()
fig3.add_trace(go.Scatter(x=dates, y=df["st_cumulative"], name="ST Cumulative",
                          line=dict(color="#e74c3c"), mode="lines+markers",
                          hovertemplate="%{x}<br>ST Cumul: $%{y:,.0f}<extra></extra>"))
fig3.add_trace(go.Scatter(x=dates, y=df["lt_cumulative"], name="LT Cumulative",
                          line=dict(color="#2ecc71"), mode="lines+markers",
                          hovertemplate="%{x}<br>LT Cumul: $%{y:,.0f}<extra></extra>"))
fig3.update_layout(title="Cumulative Realized ST / LT Gains", hovermode="x unified",
                   xaxis_title="Date", yaxis_title="$ Cumulative")
fig3.show()

# ── Chart 4: Cumulative Tax Alpha ──────────────────────────────────────────
cum_alpha = df["tax_alpha"].astype(float).cumsum() * 100
fig4 = go.Figure()
fig4.add_trace(go.Scatter(x=dates, y=cum_alpha, name="Cumul. Tax Alpha",
                          line=dict(color="#9b59b6"), mode="lines+markers",
                          fill="tozeroy", fillcolor="rgba(155,89,182,0.15)",
                          hovertemplate="%{x}<br>Tax Alpha: %{y:.3f}%<extra></extra>"))
fig4.add_hline(y=0, line_dash="dash", line_color="gray")
fig4.update_layout(title="Cumulative Tax Alpha (% of NAV)", hovermode="x unified",
                   xaxis_title="Date", yaxis_title="% of NAV")
fig4.show()


In [56]:
# Cell 7 — Lot inspection
final_date   = rebal_dates[-1]
final_prices = get_prices(final_date)

rows = [
    {
        "asset":       asset,
        "qty":         round(lot.quantity, 4),
        "basis":       round(lot.cost_basis, 2),
        "price":       round(final_prices[asset], 2),
        "unreal_gain": round((final_prices[asset] - lot.cost_basis) * lot.quantity, 2),
        "days_held":   (final_date - lot.acquisition_date).days,
        "gain_type": POLICY.classify_gain(lot, final_date, 0.0),
    }
    for asset, lots in portfolio.lots.items()
    for lot in lots
]
lot_df = pd.DataFrame(rows).sort_values("unreal_gain")
print(f"Total unrealized:  ${float(lot_df['unreal_gain'].sum()):,.0f}")
print(f"ST unrealized:     ${float(lot_df.loc[lot_df['gain_type']=='ST','unreal_gain'].sum()):,.0f}")
print(f"LT unrealized:     ${float(lot_df.loc[lot_df['gain_type']=='LT','unreal_gain'].sum()):,.0f}")
lot_df


Total unrealized:  $299,880
ST unrealized:     $0
LT unrealized:     $0


,asset,qty,basis,price,unreal_gain,days_held,gain_type
53,VO,-33.8738,291.01,296.09,-172.25,62,short_term
51,VNQ,-136.8032,90.59,90.80,-28.26,123,short_term
6,VUG,-0.0001,490.34,481.55,0.00,62,short_term
15,MLPA,0.0001,46.18,50.50,0.00,92,short_term
18,XLV,-0.0001,156.98,154.74,0.00,62,short_term
45,SHY,127.0707,82.52,82.52,0.00,0,short_term
13,VGT,-39.6913,747.92,747.92,-0.00,0,short_term
10,IVW,-0.0004,123.37,123.78,-0.00,62,short_term
36,DBC,0.0006,22.14,24.43,0.00,92,short_term
37,DBC,0.0003,22.31,24.43,0.00,62,short_term
